# visit-it — GPU validation (Colab)

Runs the three stages the CPU box could not: **MapAnything** (multi-view reconstruction),
**gsplat** (Gaussian splatting), and a **MoGe-2 GPU timing** baseline.

Takes roughly **15–30 minutes** on a free T4, most of it installing gsplat.

**Before running:** Runtime → Change runtime type → **T4 GPU**.


## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
import torch; print("torch", torch.__version__, "| cuda", torch.version.cuda, "| available", torch.cuda.is_available())

## 2. Get the code

The repo must be public for this to work unauthenticated.

In [ ]:
REPO = "https://github.com/romainbigare/visit-it.git"
BRANCH = "main"

import os, shutil
if os.path.exists("visit-it"): shutil.rmtree("visit-it")
!git clone -q --depth 1 -b {BRANCH} {REPO}
%cd visit-it
!ls -1 tools/gpu_validate_standalone.py eval/results/room_groups.json data/golden/golden_set.json

## 3. Install dependencies

Installs MoGe, MapAnything and gsplat. gsplat is the slow one — it tries a prebuilt
wheel matching Colab's torch/CUDA first and only compiles from source if that fails.

In [ ]:
# utils3d is deliberately NOT installed here: MoGe ships its own
# utils3d_moge fork, and pinning plain utils3d alongside it produces
# an unsolvable version conflict.
!pip install -q einops safetensors huggingface_hub requests
!pip install -q git+https://github.com/microsoft/MoGe.git
!pip install -q git+https://github.com/facebookresearch/map-anything.git

import sys; sys.path.insert(0, ".")
from tools.gpu_validate_standalone import install_gsplat
install_gsplat([sys.executable, "-m", "pip", "install", "--quiet"])
import gsplat; print("gsplat", gsplat.__version__)

## 4. Run the validation

Re-downloads only the images the room groups reference (~108 photos) straight from
their original URLs, so nothing needs uploading.

In [ ]:
# --max-views/--ma-max-side default to whatever the card can take;
# on a 16 GB T4 that is 3 views at 384px.
!python tools/gpu_validate_standalone.py --all \
    --max-groups 12 --iters 1500 --n-images 20 --skip-install

## 5. Look at the renders

Left is what the splat renders for a **held-out** view; right is the real photo.

In [ ]:
import glob
from IPython.display import display
from PIL import Image
pairs = sorted(glob.glob("eval/results/splat_*_render.png"))
print(f"{len(pairs)} rendered scenes")
for r in pairs[:6]:
    g = r.replace("_render.png", "_gt.png")
    ims = [Image.open(r)] + ([Image.open(g)] if os.path.exists(g) else [])
    w = sum(i.width for i in ims); h = max(i.height for i in ims)
    sheet = Image.new("RGB", (w, h), "white"); x = 0
    for i in ims: sheet.paste(i, (x, 0)); x += i.width
    print(r.split("/")[-1].replace("_render.png", ""), " (render | ground truth)")
    display(sheet)

## 6. Print the results to paste back

In [ ]:
import json, glob
for f in sorted(glob.glob("eval/results/results_*.json")):
    d = json.load(open(f))
    print("=" * 70); print(f)
    print(json.dumps({k: v for k, v in d.items() if k != "results"}, indent=2))
    for r in (d.get("results") or [])[:15]:
        print("   ", json.dumps(r))

## 7. Download everything

In [ ]:
!cd eval/results && zip -qr /content/visit-it-gpu-results.zip results_*.json splat_*.png
from google.colab import files
files.download("/content/visit-it-gpu-results.zip")